# v15: The s120/s122 mapping on the v12_robust base (replication)

**Version:** v15  
**Kaggle submission:** `v15_map_v12` → `outputs/submission_v15_map_v12.csv` (single change vs `v12_robust`, flag count matched)

**Question:** v14 gave −599 on top of the seed-averaged v13. Is that a property of the mapping or of that particular base? We repeat the same change on the best-scoring base, `v12_robust` (single-seed classifier and cost model), as a replication.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}                      # v9 cost model
DRIFT = ["s120", "s122", "s126", "s128"]; MAP = ["s120", "s122"]
keep = [s for s in S if s not in DRIFT]
f_all = tr[tr.failure == 1]; lc_all = np.log(f_all.repair_cost.to_numpy())
Xf, Xt = pd.concat([f_all[keep], shape(f_all)], axis=1), pd.concat([te[keep], shape(te)], axis=1)   # cost model: no drifted sensors (v12)
# posterior over the depot shift k from the v10 notebook (nine public scores), full precision
GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 10.0, 12.0]
W = [0.008172761879954695, 0.13116481847950048, 0.26721015776942314, 0.267056932957599, 0.1710924552535038, 0.0956936236265262,
     0.03459275261241694, 0.016549303552639335, 0.005725443007721319, 0.002368204370941116, 0.00031775367779660887,
     5.437858042612226e-05, 1.3770078232241008e-06, 3.722372808678639e-08]
pavg = lambda qq, lam=1.0: sum(w * shift(qq, k * lam) for w, k in zip(W, GRID))   # posterior-averaged shifted probability

In [5]:
def qmap(col):
    """Map each test value to the training value at the same quantile (rank-preserving)."""
    x = te[col]; r = x.rank(pct=True, method="average"); out = x.copy(); ok = x.notna()
    out[ok] = np.quantile(tr[col].dropna(), np.clip((r[ok] * ok.sum() - 0.5) / ok.sum(), 0, 1)); return out
te_map = te.copy()
for s in MAP: te_map[s] = qmap(s)

### 1. v12_robust base, scored with raw and with mapped s120/s122

In [6]:
Pl = Platt().fit(fit_oof(dev[S]), y)
mdl = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure)
q, qm = Pl.predict(mdl.predict_proba(te[S])[:, 1]), Pl.predict(mdl.predict_proba(te_map[S])[:, 1])
m, resid = cost_model(CM9, Xf, lc_all, Xt)
fl12, bu12 = decide_pv(pavg(q), m, resid)
r12 = pd.read_csv("../submissions/submission_v12_robust.csv")
assert (r12.flag.to_numpy() == fl12).all() and np.allclose(r12.budget, bu12.round(2), atol=0.01), "v12_robust not reproduced"
print(f"mean test p raw {q.mean():.4f}, mapped {qm.mean():.4f}")

mean test p raw 0.0203, mapped 0.0179


### 2. Decisions with the flag count matched to v12_robust

In [7]:
def match_flag_count(qq, m, resid, target):
    """Scale the shift (k -> lambda * k) until the number of flagged vehicles equals `target`."""
    lo, hi = 0.5, 4.0
    for _ in range(25):
        lam = (lo + hi) / 2
        if decide_pv(pavg(qq, lam), m, resid)[0].sum() < target: lo = lam
        else: hi = lam
    return lam

target = int(fl12.sum()); lam = match_flag_count(qm, m, resid, target)
fl, bu = decide_pv(pavg(qm, lam), m, resid); both = (fl12 == 1) & (fl == 1)
print(f"lambda {lam:.3f} (mean p {pavg(qm, lam).mean():.4f} vs v12 {pavg(q).mean():.4f}); flags {int(fl.sum())} (v12 {target}); "
      f"differing {int((fl12 != fl).sum())}; budget mean abs change on commonly flagged {np.abs(bu[both] - bu12[both]).mean():.1f}")
write_submission(te.id, fl, bu, "v15_map_v12")
ref = pd.read_csv("../submissions/submission_v15_map_v12.csv")
assert (ref.flag.to_numpy() == fl).all() and np.allclose(ref.budget, bu.round(2), atol=0.01), "v15 not reproduced"

lambda 1.769 (mean p 0.0359 vs v12 0.0328); flags 1760 (v12 1761); differing 269; budget mean abs change on commonly flagged 86.6
wrote outputs/submission_v15_map_v12.csv: flagged 8.80% of test vehicles


## Results and decisions

- 269 flags differ from v12_robust; mean absolute budget change ≈ 87 on commonly flagged vehicles; 1,760 flags (v12: 1,761).
- **Public score 17,058.28 vs 16,839.86 for v12_robust (+218).** The same change on the seed-averaged base was −599. Opposite signs, both within the ≈ 600-point noise of sibling files.
- **Decision:** the s120/s122 mapping has **no reliable effect** on the public score, so it is not part of the pipeline. We keep the v12 cost-model inputs and drop the mapping from further candidates; the analysis stays in the report as a hypothesis that the data neither supports nor refutes.
- **Methodological lesson:** a single public-score comparison on 30% of test (≈ 6,000 vehicles, ~1% failures) cannot resolve changes below a few hundred points. Level changes (k, budget scale) move the score by thousands and are the only ones we can read reliably.

**Caveats:** same as v14 (assumes the drift is purely instrumental; approximate isolation because flag-count matching shifts mean probabilities).